In [20]:
#Cell 1, Load raw NBA data


import pandas as pd

# Load NBA 2023 per-game stats from Basketball Reference
url = "https://www.basketball-reference.com/leagues/NBA_2023_per_game.html"
tables = pd.read_html(url)
df = tables[0]

# Remove any repeated header rows in the data
df = df[df['Player'] != 'Player']

# Convert number columns to numeric types
for col in df.columns[5:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,1.0,Joel Embiid,28.0,PHI,C,66.0,66.0,34.6,11.0,20.1,...,1.7,8.4,10.2,4.2,1.0,1.7,3.4,3.1,33.1,NaN
1,2.0,Luka Dončić,23.0,DAL,PG,66.0,66.0,36.2,10.9,22.0,...,0.8,7.8,8.6,8.0,1.4,0.5,3.6,2.5,32.4,NaN
2,3.0,Damian Lillard,32.0,POR,PG,58.0,58.0,36.3,9.6,20.7,...,0.8,4.0,4.8,7.3,0.9,0.3,3.3,1.9,32.2,NaN
3,4.0,Shai Gilgeous-Alexander,24.0,OKC,PG,68.0,68.0,35.5,10.4,20.3,...,0.9,4.0,4.8,5.5,1.6,1.0,2.8,2.8,31.4,NaN
4,5.0,Giannis Antetokounmpo,28.0,MIL,PF,63.0,63.0,32.1,11.2,20.3,...,2.2,9.6,11.8,5.7,0.8,0.8,3.9,3.1,31.1,NaN


In [21]:
#Cell 2, Clean data


# Remove duplicate player entries (e.g., players who were traded)
# We'll keep only the rows where "Tm" == "TOT" if they exist (TOT = total stats across all teams)
df = df.sort_values('Team')  # So TOT appears after the individual teams
df = df.drop_duplicates(subset='Player', keep='last')  # Keep last (usually 'TOT')

# Filter out players who played very few minutes per game (e.g., < 15 MPG)
df = df[df['MP'] >= 15]

# Reset index after filtering
df = df.reset_index(drop=True)

# Check shape and preview cleaned data
print("Final dataset shape:", df.shape)
df.head()


Final dataset shape: (342, 31)


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,110.0,Bogdan Bogdanović,30.0,ATL,SG,54.0,9.0,27.9,5.1,11.3,...,0.4,2.6,3.1,2.8,0.8,0.3,1.2,1.6,14.0,NaN
1,179.0,Onyeka Okongwu,22.0,ATL,C,80.0,18.0,23.1,4.0,6.2,...,2.7,4.5,7.2,1.0,0.7,1.3,1.0,3.1,9.9,NaN
2,49.0,Dejounte Murray,26.0,ATL,SG,74.0,74.0,36.4,8.3,17.8,...,0.7,4.5,5.3,6.1,1.5,0.3,2.2,1.4,20.5,NaN
3,90.0,De'Andre Hunter,25.0,ATL,SF,67.0,67.0,31.7,5.7,12.3,...,0.7,3.6,4.2,1.4,0.5,0.3,1.2,3.0,15.4,NaN
4,138.0,Clint Capela,28.0,ATL,C,65.0,63.0,26.6,5.4,8.2,...,4.0,7.1,11.0,0.9,0.7,1.2,0.8,2.1,12.0,NaN


In [22]:
#Cell 3, Add AllStar column


# Save the cleaned dataset to a CSV file
df.to_csv("nba_2023_cleaned.csv", index=False)

print("Data saved successfully as 'nba_2023_cleaned.csv'")


Data saved successfully as 'nba_2023_cleaned.csv'


In [23]:
#Cell 4, Machine learning to predict AllStar


# Official 2023 NBA All-Stars (both conferences, including replacements)
all_stars_2023 = [
    # East Starters
    'Kyrie Irving', 'Donovan Mitchell', 'Giannis Antetokounmpo',
    'Kevin Durant', 'Jayson Tatum',
    
    # East Reserves
    'Jaylen Brown', 'DeMar DeRozan', 'Tyrese Haliburton', 'Jrue Holiday',
    'Julius Randle', 'Bam Adebayo', 'Joel Embiid', 'Pascal Siakam',
    
    # West Starters
    'Stephen Curry', 'Luka Dončić', 'Nikola Jokić', 'LeBron James',
    'Zion Williamson',
    
    # West Reserves
    'Shai Gilgeous-Alexander', 'Damian Lillard', 'Ja Morant',
    'Paul George', 'Jaren Jackson Jr.', 'Lauri Markkanen',
    'Domantas Sabonis', 'Anthony Edwards', 'De’Aaron Fox'
]

# Label each player as 1 if they were an All-Star, 0 if not
df['AllStar'] = df['Player'].apply(lambda x: 1 if x in all_stars_2023 else 0)

# Preview
df[['Player', 'AllStar']].head(10)


,Player,AllStar
0,Bogdan Bogdanović,0
1,Onyeka Okongwu,0
2,Dejounte Murray,0
3,De'Andre Hunter,0
4,Clint Capela,0
5,Trae Young,0
6,AJ Griffin,0
7,John Collins,0
8,Robert Williams,0
9,Malcolm Brogdon,0


In [24]:
#Cell 5, Predict for custom stats/player


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Choose which stats to use as input (features)
features = ['PTS', 'AST', 'TRB', 'STL', 'BLK', 'MP']
X = df[features]

# The column we want to predict
y = df['AllStar']

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Check accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


Model Accuracy: 0.97


In [25]:
#show snubbed players


# Predict All-Star status for ALL players based on their stats
X_all = df[['PTS', 'AST', 'TRB', 'STL', 'BLK', 'MP']]
df['Predicted_AllStar'] = model.predict(X_all)

# Find players who were NOT actually All-Stars, but the model says they SHOULD be
snubs = df[(df['AllStar'] == 0) & (df['Predicted_AllStar'] == 1)]

# Show top 15 snubbed players with key stats
snubs[['Player', 'PTS', 'AST', 'TRB', 'STL', 'BLK', 'MP']].sort_values(by='PTS', ascending=False).head(15)


,Player,PTS,AST,TRB,STL,BLK,MP
25,Zach LaVine,24.8,4.2,4.5,0.9,0.2,35.9


In [26]:
#Test player


# Custom player stats (you can change these numbers)
custom_player = pd.DataFrame([{
    'PTS': 18.8,  # Points per game
    'AST': 10.5,   # Assists per game
    'TRB': 4.2,   # Total rebounds
    'STL': .8,   # Steals per game
    'BLK': .2,   # Blocks per game
    'MP': 35.4    # Minutes per game
}])

# Predict with the trained model
prediction = model.predict(custom_player)

# Show result
print("All-Star Prediction:", "⭐ YES!" if prediction[0] == 1 else "❌ Nope.")


All-Star Prediction: ❌ Nope.
